# ML System Design: Fraud Detection

Applying the universal design template to build a real-time fraud detection system.

This notebook covers:
1. System design walkthrough for fraud detection
2. Handling extreme class imbalance
3. Real-time scoring under latency constraints
4. Feature engineering (velocity, aggregates, graph, device fingerprinting)
5. Model design (GBDT + rules + anomaly detection)
6. Threshold tuning for precision/recall tradeoff
7. Concept drift and adversarial adaptation
8. Streaming architecture and monitoring

---

## Stage 1 — Problem Clarification

**Prompt**: "Design a fraud detection system for a payment platform."

### Clarifying Questions & Assumed Answers

| Question | Answer |
|----------|--------|
| What type of fraud? | Payment fraud (unauthorized transactions, stolen cards, account takeover) |
| Scale? | 10M transactions/day, ~115 TPS average, ~500 TPS peak |
| Fraud rate? | ~0.1% of transactions (1:1000 ratio — extremely imbalanced) |
| Latency? | Must score within 100ms (in the payment authorization path) |
| False positive cost? | Declined legitimate transaction → lost revenue + user friction |
| False negative cost? | Fraudulent transaction approved → chargeback cost ($25-100) + fraud loss |
| Current solution? | Rule-based system with ~200 manually written rules |
| Goal? | Reduce fraud losses by 50% without increasing false positive rate above 1% |

---
## Stage 2 — Metrics

### Why Standard Accuracy Fails

With 0.1% fraud rate, a model that predicts "not fraud" for everything achieves 99.9% accuracy. Useless.

### Offline Metrics

| Metric | Why |
|--------|-----|
| **AUC-PR** (Precision-Recall) | Primary metric — meaningful under extreme imbalance |
| **AUC-ROC** | Secondary — threshold-independent but can be misleading at low fraud rates |
| **Precision @ fixed Recall** | "At 80% recall, what is our precision?" — directly actionable |
| **Recall @ fixed FPR** | "At 1% false positive rate, what fraction of fraud do we catch?" |

### Online Metrics

| Metric | Target |
|--------|--------|
| **Fraud detection rate** (recall) | >80% of fraud caught |
| **False positive rate** | <1% of legit transactions declined |
| **$ fraud prevented** | Dollar value of blocked fraud |
| **$ legitimate declined** | Dollar value of wrongly declined transactions |
| **Review queue volume** | Number of transactions sent to manual review |

### The Economics

```
Value of catching fraud:     avg_fraud_amount * fraud_volume * detection_rate
Cost of false positives:     avg_legit_amount * legit_volume * FP_rate * customer_lifetime_value_loss
Net value = Value - Cost
```

This frames threshold tuning as an optimization problem with dollar amounts.

---
## Stage 3 — Data

### Data Sources

| Source | Examples |
|--------|----------|
| Transaction data | Amount, merchant, timestamp, currency, payment method |
| User profile | Account age, verification status, address, past behavior |
| Device data | Device fingerprint, IP address, geolocation, browser/OS |
| Session data | Login time, pages visited, time on site, actions taken |
| External data | Card BIN data, IP reputation lists, known fraud databases |
| Network/graph | Connections between users, devices, cards, merchants |

### Label Sources

1. **Chargebacks**: the definitive fraud signal, but delayed by 30-90 days
2. **Manual review decisions**: faster but introduces reviewer bias
3. **Rule-triggered blocks**: existing rule system flags, but circular (model learns the rules)
4. **User reports**: "I didn't make this transaction"

### Label Delay Problem

Chargebacks arrive 30-90 days after the transaction. This means:
- Recent training data has incomplete labels (some fraud not yet reported)
- Must use "matured" data for evaluation (at least 60 days old)
- Can use early signals (user reports, manual reviews) as proxies for faster iteration

### Sampling Strategy for Training

With 0.1% fraud rate, naive training is dominated by negatives. Strategies:
1. **Undersample** majority class (keep all fraud, sample N:1 from legitimate, e.g., 10:1)
2. **Oversample** minority class (SMOTE or random oversampling)
3. **Class weights**: weight fraud examples higher in loss function
4. **Calibration**: after resampling, recalibrate predicted probabilities

**Best practice**: undersample to ~100:1 or 50:1, use class weights, and recalibrate.

---
## Stage 4 — Feature Engineering

Feature engineering is the most critical stage for fraud detection. The model is only as good as its features.

### Velocity Features (Aggregations Over Time Windows)

| Feature | Description |
|---------|------------|
| tx_count_1h | Number of transactions by this user in last 1 hour |
| tx_count_24h | Number of transactions by this user in last 24 hours |
| tx_amount_sum_1h | Total spend by this user in last 1 hour |
| unique_merchants_24h | Number of distinct merchants in last 24 hours |
| unique_countries_1h | Number of distinct countries in last 1 hour |
| max_amount_7d | Largest single transaction in last 7 days |
| tx_amount_ratio | This amount / user's average transaction amount |

**Why velocity features matter**: Fraudsters often make many transactions quickly after compromising an account.

### Device & Network Features

| Feature | Description |
|---------|------------|
| is_new_device | First time this device is seen for this user? |
| device_age | How long has this device been associated with this user? |
| ip_risk_score | Reputation score of the IP address (VPN, proxy, Tor) |
| geo_distance_km | Distance between IP geolocation and user's registered address |
| device_user_count | Number of users associated with this device |
| is_emulator | Is the device an emulator? |

### Graph Features

Build a graph of relationships:
- User ↔ Device ↔ IP ↔ Card ↔ Merchant
- Fraud often involves shared infrastructure (same device used for multiple stolen accounts)

| Feature | Description |
|---------|------------|
| device_fraud_rate | Fraction of fraudulent transactions from this device |
| ip_fraud_rate | Fraction of fraudulent transactions from this IP |
| shared_device_count | Number of accounts using the same device |
| community_fraud_rate | Fraud rate in this user's graph neighborhood |
| degrees_from_known_fraud | Shortest path to a confirmed fraudster in the graph |

### Transaction Context Features

| Feature | Description |
|---------|------------|
| amount_zscore | How unusual is this amount for this user? |
| is_round_amount | Fraud often uses round numbers ($100, $500) |
| merchant_risk | Historical fraud rate at this merchant |
| hour_of_day | Fraud peaks at certain hours (3am local time) |
| is_first_purchase_at_merchant | First-time transactions are higher risk |

In [ ]:
import numpy as np


class VelocityFeatureComputer:
    """
    Computes velocity (time-windowed aggregate) features for fraud detection.
    
    In production, these would be computed using a streaming engine (Flink/Kafka Streams)
    backed by a time-windowed state store (Redis with TTL).
    
    This is a simplified in-memory implementation for illustration.
    """
    
    def __init__(self):
        # user_id -> list of (timestamp, amount, merchant, country)
        self.user_history = {}
    
    def add_transaction(self, user_id: str, timestamp: float, amount: float,
                        merchant: str, country: str):
        """Record a transaction."""
        if user_id not in self.user_history:
            self.user_history[user_id] = []
        self.user_history[user_id].append((timestamp, amount, merchant, country))
    
    def compute_features(self, user_id: str, current_time: float,
                         current_amount: float) -> dict:
        """Compute velocity features at the time of a new transaction."""
        history = self.user_history.get(user_id, [])
        
        # Filter by time windows
        last_1h = [(t, a, m, c) for t, a, m, c in history
                   if current_time - t <= 3600]
        last_24h = [(t, a, m, c) for t, a, m, c in history
                    if current_time - t <= 86400]
        last_7d = [(t, a, m, c) for t, a, m, c in history
                   if current_time - t <= 7 * 86400]
        
        # Compute aggregate features
        all_amounts = [a for _, a, _, _ in history] if history else [current_amount]
        avg_amount = np.mean(all_amounts)
        std_amount = np.std(all_amounts) if len(all_amounts) > 1 else avg_amount
        
        features = {
            "tx_count_1h": len(last_1h),
            "tx_count_24h": len(last_24h),
            "tx_count_7d": len(last_7d),
            "tx_amount_sum_1h": sum(a for _, a, _, _ in last_1h),
            "tx_amount_sum_24h": sum(a for _, a, _, _ in last_24h),
            "unique_merchants_24h": len(set(m for _, _, m, _ in last_24h)),
            "unique_countries_1h": len(set(c for _, _, _, c in last_1h)),
            "max_amount_7d": max((a for _, a, _, _ in last_7d), default=0),
            "amount_ratio_to_avg": current_amount / avg_amount if avg_amount > 0 else 1.0,
            "amount_zscore": (current_amount - avg_amount) / std_amount if std_amount > 0 else 0.0,
            "time_since_last_tx": (current_time - history[-1][0]) if history else 999999,
        }
        return features


# Demo
vc = VelocityFeatureComputer()

# Normal user behavior: a few transactions spread over days
base_time = 1000000
vc.add_transaction("user_A", base_time, 25.0, "coffee_shop", "US")
vc.add_transaction("user_A", base_time + 43200, 80.0, "grocery", "US")  # 12h later
vc.add_transaction("user_A", base_time + 86400, 45.0, "restaurant", "US")  # 24h later

# Suspicious: rapid transactions from different countries
vc.add_transaction("user_B", base_time, 500.0, "electronics", "US")
vc.add_transaction("user_B", base_time + 300, 800.0, "jewelry", "UK")    # 5 min later, different country!
vc.add_transaction("user_B", base_time + 600, 1200.0, "electronics", "DE")  # 10 min later, another country!

# Compute features at next transaction time
print("Normal user (user_A) — new $50 transaction:")
features_a = vc.compute_features("user_A", base_time + 90000, 50.0)
for k, v in features_a.items():
    print(f"  {k}: {v:.2f}")

print("\nSuspicious user (user_B) — new $2000 transaction:")
features_b = vc.compute_features("user_B", base_time + 900, 2000.0)
for k, v in features_b.items():
    print(f"  {k}: {v:.2f}")

---
## Stage 5 — Model Selection

### Architecture: Layered Defense

No single model catches all fraud. Use a layered approach:

```
Layer 1: Rules Engine (instant, interpretable)
  → Hard blocks: known fraudulent IPs, banned devices, velocity limits
  → Soft flags: amount > 10x average, new device + high amount

Layer 2: ML Model (GBDT — fast, strong on tabular data)
  → Scores each transaction 0-1
  → Uses all features (velocity, device, graph, context)

Layer 3: Anomaly Detection (unsupervised, catches novel fraud)
  → Isolation Forest or autoencoder on user behavior profiles
  → Flags unusual patterns that supervised model hasn't seen

Decision:
  score > 0.95 → Block
  score > 0.70 → Manual review queue
  score < 0.70 → Approve
  (thresholds tuned based on cost analysis)
```

### Why GBDT over DNN?

| Criterion | GBDT | DNN |
|-----------|------|-----|
| Tabular data performance | Excellent | Good (but not better) |
| Inference latency | < 1ms | 5-50ms |
| Interpretability | SHAP, feature importance | Low |
| Feature engineering need | Moderate | Can learn interactions |
| Training speed | Fast | Slow |

For fraud detection, GBDT (XGBoost/LightGBM) is the dominant choice because:
- Latency is critical (in the payment path)
- Features are tabular
- Interpretability matters (regulatory requirements, manual review)
- Handles missing values and mixed types natively

---
## Threshold Tuning: Precision-Recall Tradeoff

In [ ]:
import numpy as np


def simulate_fraud_scores(n_legit=100000, n_fraud=100, seed=42):
    """
    Simulate model scores for legitimate and fraudulent transactions.
    Fraud ratio ~0.1% (realistic).
    """
    np.random.seed(seed)
    # Legitimate: most get low scores, some overlap with fraud
    scores_legit = np.random.beta(2, 10, n_legit)  # skewed low
    # Fraud: most get high scores, some are hard to detect
    scores_fraud = np.random.beta(5, 2, n_fraud)    # skewed high
    
    scores = np.concatenate([scores_legit, scores_fraud])
    labels = np.concatenate([np.zeros(n_legit), np.ones(n_fraud)])
    return scores, labels


def precision_recall_at_thresholds(scores, labels, thresholds):
    """Compute precision, recall, and FPR at each threshold."""
    results = []
    for t in thresholds:
        predicted_positive = scores >= t
        tp = np.sum(predicted_positive & (labels == 1))
        fp = np.sum(predicted_positive & (labels == 0))
        fn = np.sum(~predicted_positive & (labels == 1))
        tn = np.sum(~predicted_positive & (labels == 0))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        
        results.append({
            "threshold": t,
            "precision": precision,
            "recall": recall,
            "fpr": fpr,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn
        })
    return results


def cost_analysis(results, avg_fraud_amount=500, avg_legit_amount=50,
                  chargeback_cost=30, customer_loss_prob=0.1,
                  customer_ltv=1000):
    """
    Compute dollar impact for each threshold.
    
    Cost of missing fraud (FN): fraud_amount + chargeback_cost
    Cost of false positive (FP): customer may leave (LTV loss) + immediate revenue loss
    Value of catching fraud (TP): fraud_amount saved + chargeback avoided
    """
    for r in results:
        fraud_caught_value = r["tp"] * (avg_fraud_amount + chargeback_cost)
        fraud_missed_cost = r["fn"] * (avg_fraud_amount + chargeback_cost)
        fp_cost = r["fp"] * (avg_legit_amount + customer_loss_prob * customer_ltv)
        
        r["net_value"] = fraud_caught_value - fraud_missed_cost - fp_cost
        r["fraud_caught_value"] = fraud_caught_value
        r["fraud_missed_cost"] = fraud_missed_cost
        r["fp_cost"] = fp_cost
    return results


# Run analysis
scores, labels = simulate_fraud_scores()
thresholds = np.arange(0.1, 0.95, 0.05)
results = precision_recall_at_thresholds(scores, labels, thresholds)
results = cost_analysis(results)

print(f"{'Threshold':>9} {'Precision':>9} {'Recall':>8} {'FPR':>6} {'FP':>6} {'FN':>4} {'Net Value ($)':>13}")
print("-" * 65)
for r in results:
    print(f"{r['threshold']:>9.2f} {r['precision']:>9.3f} {r['recall']:>8.3f} "
          f"{r['fpr']:>6.4f} {r['fp']:>6} {r['fn']:>4} {r['net_value']:>13,.0f}")

# Find optimal threshold
best = max(results, key=lambda r: r["net_value"])
print(f"\nOptimal threshold: {best['threshold']:.2f}")
print(f"  Precision: {best['precision']:.3f}, Recall: {best['recall']:.3f}")
print(f"  FPR: {best['fpr']:.4f}, Net Value: ${best['net_value']:,.0f}")

The key insight: the optimal threshold depends on the **business economics**, not just ML metrics. Different cost assumptions lead to different optimal thresholds.

---
## Handling Extreme Class Imbalance

In [ ]:
import numpy as np


def demonstrate_imbalance_strategies():
    """
    Show how different sampling and weighting strategies affect
    a simple logistic regression on imbalanced data.
    """
    np.random.seed(42)
    
    # Generate imbalanced data: 0.1% positive rate
    n_neg, n_pos = 10000, 10
    X_neg = np.random.randn(n_neg, 5) * 1.0
    X_pos = np.random.randn(n_pos, 5) * 1.0 + 2.0  # shifted mean
    
    X = np.vstack([X_neg, X_pos])
    y = np.concatenate([np.zeros(n_neg), np.ones(n_pos)])
    
    # Simple logistic regression with gradient descent
    def sigmoid(z):
        return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))
    
    def train_lr(X, y, weights=None, lr=0.01, n_iter=500):
        if weights is None:
            weights = np.ones(len(y))
        w = np.zeros(X.shape[1])
        b = 0.0
        for _ in range(n_iter):
            p = sigmoid(X @ w + b)
            error = (p - y) * weights
            w -= lr * (X.T @ error) / len(y)
            b -= lr * np.mean(error)
        return w, b
    
    def evaluate(X, y, w, b):
        p = sigmoid(X @ w + b)
        pred = (p > 0.5).astype(int)
        tp = np.sum((pred == 1) & (y == 1))
        fp = np.sum((pred == 1) & (y == 0))
        fn = np.sum((pred == 0) & (y == 1))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        return precision, recall, tp, fp, fn
    
    # Strategy 1: No adjustment
    w1, b1 = train_lr(X, y)
    p1, r1, tp1, fp1, fn1 = evaluate(X, y, w1, b1)
    print("Strategy 1: No adjustment")
    print(f"  Precision: {p1:.3f}, Recall: {r1:.3f} (TP={tp1}, FP={fp1}, FN={fn1})")
    
    # Strategy 2: Class weights (weight positives by n_neg/n_pos)
    sample_weights = np.where(y == 1, n_neg / n_pos, 1.0)
    w2, b2 = train_lr(X, y, weights=sample_weights)
    p2, r2, tp2, fp2, fn2 = evaluate(X, y, w2, b2)
    print("\nStrategy 2: Class weights (1000:1)")
    print(f"  Precision: {p2:.3f}, Recall: {r2:.3f} (TP={tp2}, FP={fp2}, FN={fn2})")
    
    # Strategy 3: Undersampling (keep all positives, subsample negatives)
    ratio = 10  # 10:1 negative:positive
    neg_idx = np.random.choice(n_neg, n_pos * ratio, replace=False)
    X_under = np.vstack([X_neg[neg_idx], X_pos])
    y_under = np.concatenate([np.zeros(n_pos * ratio), np.ones(n_pos)])
    w3, b3 = train_lr(X_under, y_under)
    p3, r3, tp3, fp3, fn3 = evaluate(X, y, w3, b3)
    print(f"\nStrategy 3: Undersampling (10:1 ratio)")
    print(f"  Precision: {p3:.3f}, Recall: {r3:.3f} (TP={tp3}, FP={fp3}, FN={fn3})")


demonstrate_imbalance_strategies()

---
## Concept Drift and Adversarial Adaptation

### Why Fraud Models Decay Faster Than Other ML Models

Fraud is adversarial: fraudsters **actively adapt** to the model.

```
Cycle:
1. Model deployed, catches fraud pattern X
2. Fraudsters notice pattern X is blocked
3. Fraudsters switch to pattern Y
4. Model misses pattern Y (never seen in training data)
5. New fraud losses increase
6. Model retrained with pattern Y data
7. Go to step 1 with pattern Y
```

### Mitigation Strategies

| Strategy | How |
|----------|-----|
| **Frequent retraining** | Daily or even hourly model updates |
| **Anomaly detection** | Unsupervised models catch novel patterns without labeled data |
| **Rules as fast response** | Deploy new rules within hours for emerging patterns |
| **Feature robustness** | Use features that are hard for fraudsters to manipulate (graph features, behavioral biometrics) |
| **Ensemble diversity** | Multiple models with different feature sets — harder to evade all simultaneously |
| **Online learning** | Continuously update model weights as new labeled data arrives |

### Concept Drift Detection

- Monitor fraud rate by model score bucket (should be monotonically increasing)
- Track PSI (Population Stability Index) on key features weekly
- Monitor the model's ROC-AUC on the most recent "matured" data
- Set up alerts for sudden changes in fraud patterns (new merchant categories, new geographies)

---
## Stage 7 — Streaming Architecture

Fraud detection must be real-time (in the payment authorization path).

### Architecture

```
Transaction Event
    ↓
Kafka Topic (raw_transactions)
    ↓
Feature Service
    ├── Real-time features (velocity) ← Redis (time-windowed aggregates)
    ├── Precomputed features (user profile) ← Feature Store
    └── External lookups (IP reputation) ← API cache
    ↓
Rules Engine (Layer 1)
    → Hard block / pass-through
    ↓
ML Scoring Service (Layer 2)
    → XGBoost model, <5ms inference
    ↓
Decision Service
    → Approve / Decline / Review
    ↓
Kafka Topic (decisions)
    ↓
Logging + Feedback Pipeline
```

### Latency Breakdown

```
Event ingestion:        5ms
Feature computation:    30ms (Redis lookups + real-time aggregates)
Rules engine:           5ms
ML scoring:             5ms
Decision + logging:     5ms
---
Total:                 ~50ms (well within 100ms budget)
```

### Feature Store for Fraud

- **Online (Redis)**: velocity features, user profile, device fingerprint
  - Velocity features use Redis sorted sets with TTL for time-windowed aggregates
  - Example: `ZADD user:123:tx_timestamps <timestamp> <tx_id>` + `ZRANGEBYSCORE` for window queries
- **Offline (Hive/S3)**: historical aggregates, graph features, training data
  - Computed daily by Spark jobs
  - Materialized to Redis for online serving

---
## Stage 8 — Monitoring

### What to Monitor

| Metric | Frequency | Alert Threshold |
|--------|-----------|----------------|
| Fraud detection rate (on matured data) | Daily | < 70% (was 80%) |
| False positive rate | Hourly | > 1.5% |
| Model score distribution | Hourly | PSI > 0.1 |
| Feature null rates | Real-time | Any feature > 5% null |
| Scoring latency (p99) | Real-time | > 50ms |
| Review queue volume | Hourly | > 2x normal |
| Fraud dollar loss (daily) | Daily | > 1.5x trailing 7-day average |

### Alert Fatigue

A critical problem in fraud detection: too many false alarms cause analysts to ignore alerts.

**Mitigation**:
- Prioritize the review queue by model confidence (highest risk first)
- Track analyst override rate (if they approve >80% of flagged transactions, threshold is too low)
- Use tiered alerting: auto-block high confidence, review medium confidence, log low confidence
- Feedback loop: analyst decisions feed back into model training

### Feedback Loops

**Problem**: If the model blocks a transaction, we never learn the true label (would it have been fraud?).

**Solution**: Periodically allow a small random sample of blocked transactions through ("exploration"), or use inverse propensity scoring to account for selection bias in training data.

---
## Key Interview Talking Points

1. **Class imbalance is the defining challenge**: know at least 3 strategies (undersampling, class weights, focal loss) and when to use each.
2. **Precision-recall tradeoff is a business decision**: frame it in dollar terms, not just ML metrics.
3. **Layered defense**: rules + ML + anomaly detection. No single model catches all fraud.
4. **Adversarial adaptation**: the world changes because fraudsters react. Discuss frequent retraining, anomaly detection, and feature robustness.
5. **Real-time constraints**: know the streaming architecture (Kafka + Redis + model server). Discuss latency budgets.
6. **Label delay**: chargebacks take 30-90 days. Discuss how this affects training and evaluation.
7. **Feature engineering dominates**: velocity features, graph features, and device fingerprinting are often more impactful than model architecture choices.
8. **Monitoring and alert fatigue**: a unique challenge in fraud. Discuss analyst workflow and feedback loops.